In [44]:
from test_dit_base import *

In [46]:
model = torch.load('wandb/run-20240922_171844-05pywxxs/files/model-5000000.pt', map_location='cpu').to(1).eval()

In [47]:
rank = 0
max_timestep = 512
sampling_step = 50
bs = 4
datasetinfo = VideoDatasetInfo(is_latent=False, image_shape=(1,16,64,64))
img = sample_image(1, model, 512, sampling_step, bs, None, datasetinfo, True, 1)

In [48]:
import math
from einops import rearrange
recon_video = img
iiii = int(math.sqrt(bs))
recon_video = rearrange(recon_video.reshape((iiii, bs//iiii, *recon_video.shape[1:])), 'bh bw c t h w -> c t (bh h) (bw w)')

In [49]:
recon_video.shape

torch.Size([1, 16, 128, 128])

In [51]:
from IPython.display import Video, display
import torchvision
@torch.no_grad
def display_video_tensor(video_tensor, fps=4, filename='temp_video.mp4'):
    # Ensure tensor is on CPU
    video_tensor = torch.clip(video_tensor + 0.5 , 0, 1)
    video = video_tensor.cpu()
    
    # Ensure the tensor is in the correct format (T, C, H, W)
    if video.ndim == 3:
        video = video.unsqueeze(1)  # Add channel dimension if it's missing
    elif video.shape[1]  in (1,3) and video.ndim == 4:
        video = video.permute(0, 2, 3, 1)  # Change from (T, C, H, W) to (T, H, W, C)
    elif video.shape[0] in (1,3)  and video.ndim == 4:
        video = video.permute(1, 2, 3, 0) 
    if video.shape[-1] == 1 and video.ndim == 4:
        video = video.repeat(1,1,1, 3)
    # Ensure the video is in uint8 format
    video = (video * 255).to(torch.uint8)
    # Write video to a file
    print(video.shape)
    torchvision.io.write_video(filename=filename, video_array=video, fps=fps)
    # Display the video
    return Video(filename, embed=True,width=256, height=256)
display_video_tensor(recon_video)

torch.Size([16, 128, 128, 3])
